# Contextual Chunking — kiểm chứng tính công bằng của khóa Sparse

Phase 1 so sánh **retriever**, nhưng luôn trên **một cách biểu diễn dữ liệu duy nhất**:
mọi chỉ mục đều xây từ `chunk["text"]` và không gì khác. Vậy kết luận *"sparse thắng
dense"* thực chất là *"sparse thắng dense **trên cách biểu diễn này**"*.

Notebook này thay đổi cách biểu diễn rồi đo lại. Toàn bộ nằm trong đây — kỹ thuật dữ liệu,
kỹ thuật đặc trưng, dựng chỉ mục, đánh giá, kiểm định thống kê.

---
## Giả thuyết

Bài báo trung bình cắt thành **2,06 chunk**. Chunk đầu tiên mở đầu bằng chính bài báo
nên tự nó đã có ngữ cảnh. Nhưng các chunk tiếp theo — **51,4% toàn kho** — đến tay
retriever mà **không có dấu hiệu nào cho biết chúng thuộc câu chuyện gì**:

```
chunk 0  "LOS ANGELES (CNN) -- Natalie Cole's search for a new kidney ended..."
chunk 1  "...she said. The transplant was performed Tuesday. Doctors said..."   <- ngữ cảnh?
```

**Can thiệp:** nối đoạn mở đầu bài báo vào trước mỗi chunk tiếp nối.

> ### Một chuyện phải nói trước
> Ý tưởng ban đầu là *"embed thêm metadata/title"*. **Không làm được** — dataset này
> không có metadata thật. `metadata.title` **không phải headline**, nó là **160 ký tự
> đầu của chính bài báo** (đã kiểm: là tiền tố của body ở 200/200 bài đánh giá).
> `url`, `author`, `publish_date` đều là chuỗi rỗng do chunker gán mặc định;
> `publisher` là `"CNN"` cho cả 11.064 bài. Cell 3 sẽ in ra để ông tự thấy.
>
> Thứ duy nhất có giá trị trong trường đó là **đoạn mở đầu bài báo** — nên thí nghiệm
> này là *contextual chunking*, không phải *metadata indexing*. Tên gọi khác nhau,
> và báo cáo phải gọi đúng tên.

> ### Còn trục query prompt thì sao?
> **Đã làm rồi, đúng chuẩn.** `embeddings.py:131-144` áp prefix chính tắc của từng mô hình:
> `query: `/`passage: ` cho e5, và instruction riêng của BGE cho bge-*. bge-m3 đúng ra
> không dùng prefix nào và code cũng không thêm. Nên nhánh dense **chưa bao giờ ngây thơ**
> ở trục đó — thêm prompt tự chế lên trên là đi lệch khỏi phân phối huấn luyện, nhiều
> khả năng làm tệ đi.

---
## Notebook này khác lần chạy Phase 1 ở một điểm quan trọng

Nhánh dense ở đây dùng **tìm kiếm chính xác** (nhân ma trận trên vector đã chuẩn hóa),
**không dùng Chroma/HNSW**. Hai lợi ích:

1. **Tái lập được.** HNSW là chỉ mục *xấp xỉ* và không được truyền seed — chính nguồn
   nhiễu khiến số dense của Phase 1 trôi tới 0,0143 giữa hai lần chạy. Ở đây không có.
2. **Công bằng hơn với dense.** Tìm kiếm chính xác không mất recall do xấp xỉ, nên đây
   là baseline dense **mạnh hơn** so với lần chạy tournament.

Nghĩa là nếu sparse vẫn thắng ở đây, kết luận còn vững hơn trước.


---
## 1. Cấu hình

Bật GPU (T4). Bật Internet. Không cần secret nào (dataset công khai).

Chạy thử trước với `SMOKE = True` (~5 phút) để chắc không lỗi, rồi đổi thành `False`.


In [ ]:
SMOKE = True          # True: 40 câu + 3.000 chunk. False: chạy đầy đủ.

REPO_URL    = 'https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
# Phải là SHA đủ 40 ký tự: 'git fetch' không nhận SHA rút gọn.
REPO_COMMIT = '2735bdc62df371cbdf53e9feff6b9d5cc5524a3f'   # hoặc 'main' để lấy bản mới nhất
HF_REPO_ID  = 'MatchaMacchiato/newsqa_200_11064_v2.0.0'
HF_REVISION = 'b81c8db6847a23272665946c0c43c72e9a212fd9'

# --- Kỹ thuật đặc trưng: nút vặn nằm ở đây ---
CONTEXT_CHARS   = 160     # bao nhiêu ký tự đầu bài báo dùng làm ngữ cảnh
CONTEXT_SEP     = '\n\n' # ngăn cách ngữ cảnh với thân chunk
SKIP_FIRST_CHUNK = True   # chunk 0 đã mở đầu bài báo -> không nối lại (tránh lặp)

DENSE_MODEL  = 'intfloat/e5-base-v2'
SPARSE_MODEL = 'BAAI/bge-m3'
TOP_K        = 10
K_VALUES     = [1, 3, 5, 10]
VARIANTS     = ['resolved', 'original']

BOOTSTRAP_SAMPLES, SEED = 1000, 42


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
PROJECT = WORK / 'Text-Mining---NewsQA-RAG'
DATA, OUT = WORK / 'ablation_data', WORK / 'ablation_results'
OUT.mkdir(parents=True, exist_ok=True)
os.environ.update({'TOKENIZERS_PARALLELISM':'false','HF_HOME':str(WORK/'hf_cache'),
                   'OMP_NUM_THREADS':'1','PYTHONUNBUFFERED':'1'})

if not PROJECT.exists():
    subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT)],check=True)

def checkout(ref):
    """Lấy đúng ref. Fetch SHA trực tiếp chỉ chạy với SHA đủ 40 ký tự và khi
    server cho phép; nếu không thì fetch cả nhánh rồi checkout."""
    if ref == 'main':
        subprocess.run(['git','fetch','origin','main'],cwd=PROJECT,check=True)
        subprocess.run(['git','checkout','--detach','origin/main'],cwd=PROJECT,check=True)
        return
    if subprocess.run(['git','fetch','--depth=1','origin',ref],cwd=PROJECT).returncode != 0:
        print(f'fetch thẳng {ref[:12]} không được — tải cả lịch sử rồi checkout')
        subprocess.run(['git','fetch','--unshallow','origin'],cwd=PROJECT)
        subprocess.run(['git','fetch','origin'],cwd=PROJECT,check=True)
    subprocess.run(['git','checkout','--detach',ref],cwd=PROJECT,check=True)

checkout(REPO_COMMIT)
print('repo tại', subprocess.run(['git','rev-parse','--short','HEAD'],cwd=PROJECT,
                                 capture_output=True,text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],
               cwd=PROJECT,check=True)
sys.path.insert(0, str(PROJECT/'common'))

import numpy as np, pandas as pd, torch
from newsqa_rag.evaluation.metrics import evaluate_retrieval
from newsqa_rag.evaluation.metrics import hit_rate_at_k, mrr_at_k, ndcg_at_k, recall_at_k
from newsqa_rag.experiments import paired_comparison
from newsqa_rag.indexing.learned_sparse_index import BGEM3SparseEncoder, LearnedSparseIndex

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')


---
## 2. Lấy dữ liệu

Tải bundle v2.0.0 đã khóa (kho đã phục hồi phần đuôi bài báo, 22.766 chunk).


In [ ]:
if not (DATA/'newsqa_200_11064/final_deduplicated/chunks.jsonl').exists():
    subprocess.run([sys.executable,'scripts/materialize_evaluation_dataset.py',
                    '--repo-id',HF_REPO_ID,'--revision',HF_REVISION,
                    '--output-root',str(DATA/'newsqa_200_11064'),
                    '--db-path',str(DATA/'chroma'),'--skip-vector-index'],
                   cwd=PROJECT,check=True)

ROOT  = DATA/'newsqa_200_11064'
FINAL = ROOT/'final_deduplicated'

def read_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text(encoding='utf-8').splitlines() if l.strip()]

chunks   = read_jsonl(FINAL/'chunks.jsonl')
testsets = {'resolved': read_jsonl(FINAL/'testset_resolved.jsonl'),
            'original': read_jsonl(FINAL/'testset_reviewed_original.jsonl')}
print(f'{len(chunks):,} chunk')
for name, rows in testsets.items():
    print(f'  {name:9s} {len(rows):,} câu hỏi')


---
## 3. Kỹ thuật dữ liệu — trong dataset này thực sự có gì?

Trước khi xây đặc trưng, kiểm tra metadata. Đây là cell chứng minh vì sao
*"embed metadata"* không khả thi.


In [ ]:
meta_keys = sorted({k for c in chunks for k in c['metadata']})
print('Các trường metadata và độ đa dạng của chúng:\n')
for k in meta_keys:
    vals = [str(c['metadata'].get(k,'')) for c in chunks]
    nonempty = sum(1 for v in vals if v.strip())
    uniq = len(set(vals))
    verdict = ('RỖNG' if nonempty == 0 else
               'HẰNG SỐ (0 thông tin)' if uniq == 1 else
               f'{uniq:,} giá trị khác nhau')
    print(f'  {k:22s} có giá trị: {nonempty:6,}/{len(chunks):,}   {verdict}')

# title có phải headline không, hay chỉ là phần đầu của body?
is_prefix = sum(1 for c in chunks
                if (t:=str(c['metadata'].get('title','')).strip())
                and c['text'].lstrip().startswith(t))
first_chunks = sum(1 for c in chunks if c['metadata'].get('chunk_index') == 0)
print(f'\ntitle là tiền tố của chính text: {is_prefix:,} chunk '
      f'(và có {first_chunks:,} chunk đầu bài)')
print('\nVí dụ "title":')
for c in chunks[:3]:
    print(f'  {str(c["metadata"]["title"])[:95]!r}')
print('\n=> Đây là 160 ký tự đầu của bài báo, KHÔNG phải headline.')
print('=> Không có metadata nào để index. Nhưng có ngữ cảnh cấp bài báo để tận dụng.')


---
## 4. Kỹ thuật đặc trưng — dựng kho ngữ liệu có ngữ cảnh

**Quy tắc bất di bất dịch:** chỉ đổi `text`. **Không đụng `id`** — nhãn vàng trỏ vào
id, và mọi so sánh phía dưới đều ghép cặp theo id. Đổi id là hỏng toàn bộ thí nghiệm.


In [ ]:
def build_contextual(chunks, context_chars=CONTEXT_CHARS, sep=CONTEXT_SEP,
                     skip_first=SKIP_FIRST_CHUNK):
    """Nối ngữ cảnh cấp bài báo vào trước mỗi chunk tiếp nối."""
    out, changed = [], 0
    for c in chunks:
        text = c['text']
        lead = str(c['metadata'].get('title','')).strip()[:context_chars]
        is_first = c['metadata'].get('chunk_index') == 0
        already  = lead and text.lstrip().startswith(lead)
        if lead and not (skip_first and is_first) and not already:
            text = f'{lead}{sep}{text}'
            changed += 1
        out.append({**c, 'text': text})
    return out, changed

contextual, changed = build_contextual(chunks)

assert [a['id'] for a in chunks] == [b['id'] for b in contextual], 'ID phải giữ nguyên'
assert changed > 0, 'không chunk nào được thêm ngữ cảnh — kiểm tra lại CONTEXT_CHARS'
print(f'Đã thêm ngữ cảnh cho {changed:,}/{len(chunks):,} chunk ({changed/len(chunks):.1%})')

ex = next(i for i,(a,b) in enumerate(zip(chunks,contextual)) if a['text'] != b['text'])
print(f'\n--- Ví dụ ({chunks[ex]["id"]}) ---')
print('TRƯỚC:', repr(chunks[ex]['text'][:150]))
print('SAU  :', repr(contextual[ex]['text'][:150]))


---
## 5. Thu nhỏ mẫu khi chạy thử

Khi `SMOKE`, giữ lại toàn bộ chunk vàng của các câu được chọn rồi mới bù thêm chunk
nhiễu — nếu không, chunk đúng bị loại khỏi kho và mọi chỉ số bằng 0.


In [ ]:
if SMOKE:
    for name in VARIANTS:
        testsets[name] = testsets[name][:40]
    gold = {cid for rows in testsets.values() for r in rows
            for cid in (r.get('relevant_chunk_ids') or [])}
    keep, extra = [], 0
    for a, b in zip(chunks, contextual):
        if a['id'] in gold:
            keep.append((a,b))
        elif extra < 3000:
            keep.append((a,b)); extra += 1
    chunks     = [a for a,_ in keep]
    contextual = [b for _,b in keep]
    print(f'SMOKE: {len(chunks):,} chunk ({len(gold)} chunk vàng + {extra:,} nhiễu), '
          f'{len(testsets["resolved"])} câu/biến thể')
else:
    print(f'ĐẦY ĐỦ: {len(chunks):,} chunk, {len(testsets["resolved"]):,} câu/biến thể')

CORPORA = {'plain': chunks, 'contextual': contextual}


---
## 6. Nhánh Dense — tìm kiếm chính xác, không dùng HNSW

Mã hóa toàn bộ chunk một lần, giữ trong một ma trận, rồi tính tương đồng bằng phép nhân.
Vector đã chuẩn hóa nên tích vô hướng chính là cosine.

Prefix `passage: ` / `query: ` là **bắt buộc với e5** — đó là cách mô hình được huấn luyện.
Bỏ nó đi là làm hỏng mô hình chứ không phải "thử nghiệm công bằng".


In [ ]:
from sentence_transformers import SentenceTransformer

_dense = SentenceTransformer(DENSE_MODEL, device=DEVICE)

def dense_rank(corpus, queries, batch_size=64):
    """Trả về top-K id cho mỗi câu hỏi, bằng tìm kiếm chính xác."""
    ids = [c['id'] for c in corpus]
    docs = _dense.encode([f'passage: {c["text"]}' for c in corpus],
                         batch_size=batch_size, normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=True)
    qs = _dense.encode([f'query: {q}' for q in queries],
                       batch_size=batch_size, normalize_embeddings=True,
                       convert_to_numpy=True)
    sims = qs @ docs.T                       # (n_queries, n_chunks)
    top = np.argsort(-sims, axis=1)[:, :TOP_K]
    return [[ids[j] for j in row] for row in top]

print('dense sẵn sàng — tìm kiếm chính xác, không xấp xỉ, tái lập được')


---
## 7. Nhánh Sparse — BGE-M3 learned sparse

Dùng đúng lớp chỉ mục của repo. bge-m3 **không dùng** prefix hướng dẫn nào — đúng chuẩn.


In [ ]:
_sparse_encoder = BGEM3SparseEncoder(model_name=SPARSE_MODEL, device=DEVICE)

def sparse_rank(corpus, queries, batch_size=16):
    # encoder truyền qua constructor -> model chỉ nạp một lần cho cả 8 nhánh
    index = LearnedSparseIndex(model_name=SPARSE_MODEL, device=DEVICE,
                               encoder=_sparse_encoder)
    t0 = time.time()
    index.build(corpus, batch_size=batch_size)
    print(f'    dựng chỉ mục sparse trong {time.time()-t0:.0f}s', flush=True)
    return [[hit['id'] for hit in index.query(q, top_k=TOP_K)] for q in queries]

print('sparse sẵn sàng')


---
## 8. Chạy toàn bộ 8 nhánh

2 retriever × 2 kho ngữ liệu × 2 biến thể câu hỏi. Đây là phần lâu nhất.


In [ ]:
RANKERS = {'dense': dense_rank, 'sparse': sparse_rank}
per_question, aggregate = {}, []

for retriever, rank_fn in RANKERS.items():
    for corpus_name, corpus in CORPORA.items():
        for variant in VARIANTS:
            rows = testsets[variant]
            key = (retriever, corpus_name, variant)
            print(f'\n>>> {retriever} / {corpus_name} / {variant}', flush=True)
            t0 = time.time()
            ranked = rank_fn(corpus, [r['question'] for r in rows])

            samples = [{'question_id': r['question_id'],
                        'relevant_chunk_ids': r.get('relevant_chunk_ids') or [],
                        'retrieved_ids': ids}
                       for r, ids in zip(rows, ranked)]
            per_question[key] = [
                {'question_id': s['question_id'],
                 'retrieval': {f'{fn.__name__.replace("_at_k","")}@{k}':
                               fn(s['relevant_chunk_ids'], s['retrieved_ids'], k)
                               for fn in (hit_rate_at_k, mrr_at_k, ndcg_at_k, recall_at_k)
                               for k in K_VALUES}}
                for s in samples]

            summary = evaluate_retrieval(samples, K_VALUES)
            aggregate.append({'retriever': retriever, 'corpus': corpus_name,
                              'variant': variant, **summary})
            print(f'    nDCG@5={summary["ndcg@5"]:.4f}  Hit@1={summary["hit_rate@1"]:.4f}  '
                  f'Hit@5={summary["hit_rate@5"]:.4f}  ({time.time()-t0:.0f}s)', flush=True)

df = pd.DataFrame(aggregate)
display(df[['retriever','corpus','variant','ndcg@5','mrr@5','hit_rate@1','hit_rate@5']])


---
## 9. Kiểm định theo cặp — contextual có thật sự giúp không?

Mọi nhánh chạy trên **cùng bộ câu hỏi**, nên độ khó của câu hỏi là nhiễu dùng chung và
triệt tiêu khi lấy hiệu theo từng câu. **CI95 của hiệu không chứa 0 ⇒ có ý nghĩa.**


In [ ]:
results = []
for retriever in RANKERS:
    for variant in VARIANTS:
        base = per_question[(retriever,'plain',variant)]
        ctx  = per_question[(retriever,'contextual',variant)]
        row = {'retriever': retriever, 'variant': variant}
        for metric in ('retrieval.ndcg@5','retrieval.mrr@5',
                       'retrieval.hit_rate@1','retrieval.hit_rate@5'):
            out = paired_comparison(base, ctx, metric, BOOTSTRAP_SAMPLES, SEED)
            if out:
                sig = out['ci95_low'] > 0 or out['ci95_high'] < 0
                row[metric.replace('retrieval.','')] = (
                    f"{out['mean_delta_right_minus_left']:+.4f} "
                    f"[{out['ci95_low']:+.4f},{out['ci95_high']:+.4f}]"
                    + ('  CO Y NGHIA' if sig else ''))
        results.append(row)

print('Hiệu = contextual trừ plain. Dương = ngữ cảnh giúp ích.\n')
display(pd.DataFrame(results))


---
## 10. Câu hỏi quyết định: khoảng cách sparse–dense có bị thu hẹp không?

Đây mới là điều thật sự quan trọng. Contextual chunking giúp **cả hai** nhánh — nếu nó
giúp dense nhiều hơn thì khóa sparse cần xem lại; nếu không, khóa được củng cố.


In [ ]:
print('Khoảng cách sparse − dense theo nDCG@5 (dương = sparse dẫn trước)\n')
gap_rows = []
for corpus_name in CORPORA:
    for variant in VARIANTS:
        d = paired_comparison(per_question[('dense',corpus_name,variant)],
                              per_question[('sparse',corpus_name,variant)],
                              'retrieval.ndcg@5', BOOTSTRAP_SAMPLES, SEED)
        gap_rows.append({'corpus': corpus_name, 'variant': variant,
                         'gap': round(d['mean_delta_right_minus_left'],4),
                         'ci95': f"[{d['ci95_low']:+.4f},{d['ci95_high']:+.4f}]",
                         'sparse_thang_co_y_nghia': d['ci95_low'] > 0})
gaps = pd.DataFrame(gap_rows)
display(gaps)

print('\n' + '='*70)
for variant in VARIANTS:
    p = gaps[(gaps.corpus=='plain') & (gaps.variant==variant)]['gap'].iloc[0]
    c = gaps[(gaps.corpus=='contextual') & (gaps.variant==variant)]['gap'].iloc[0]
    print(f'{variant:9s}  plain {p:+.4f}  ->  contextual {c:+.4f}   '
          f'(thay đổi {c-p:+.4f})')
print('='*70)
print('\nKhoảng cách THU HẸP đáng kể  -> đáng xem lại khóa, báo cáo phải nêu.')
print('Khoảng cách GIỮ NGUYÊN/RỘNG  -> khóa sparse được củng cố bằng bằng chứng.')


---
## 11. Lưu kết quả


In [ ]:
payload = {
    'smoke': SMOKE,
    'config': {'context_chars': CONTEXT_CHARS, 'skip_first_chunk': SKIP_FIRST_CHUNK,
               'dense_model': DENSE_MODEL, 'sparse_model': SPARSE_MODEL,
               'top_k': TOP_K, 'seed': SEED, 'bootstrap_samples': BOOTSTRAP_SAMPLES,
               'dense_search': 'exact matrix product (no HNSW, reproducible)',
               'n_chunks': len(chunks), 'n_contextualised': changed},
    'aggregate': aggregate,
    'contextual_effect': results,
    'sparse_dense_gap': gap_rows,
}
path = OUT/'contextual_chunking_results.json'
path.write_text(json.dumps(payload, indent=2, ensure_ascii=False)+'\n', encoding='utf-8')
print('Đã ghi', path)
print('\nGửi file này lại để cập nhật báo cáo.')
if SMOKE:
    print('\n*** Đây mới là SMOKE. Đặt SMOKE = False rồi chạy lại để lấy số thật. ***')
